# Week 3 Notebook Lecture: Data Structures for Workplace Data

**Course:** DSSA 5102 Data Gathering and Warehousing  
**Week:** September 24  
**Environment:** Google Colab free tier or local Jupyter

This tutorial uses a realistic facilities and service-request scenario to compare dictionaries, flat tables, normalized relational tables, nested JSON, and append-only events. The objective is not to identify one universally best structure. The objective is to select a structure that matches the work people need to perform.

The notebook follows the course evidence loop: **define, build, inspect, verify, compare, and document**.

## Runtime and Data Note

This notebook uses standard Python plus `pandas`, which is normally available in Google Colab. It requires no shell commands, external credentials, paid service, GPU, or large download. Expected runtime is under five minutes.

All records are synthetic. The names, buildings, identifiers, and events do not describe real people. This makes the examples safe to share while preserving realistic data-management problems. Files saved in Colab are temporary unless downloaded or copied to Google Drive.

## Learning Objectives

By the end of the notebook, students should be able to:

- Represent one business record with a Python dictionary.
- Convert a list of records into a pandas table.
- Explain why flat tables duplicate some information.
- Separate shared entities into normalized relational tables.
- Represent a complete object and its child records as nested JSON.
- Preserve change history as an append-only event stream.
- Reconstruct current state by replaying events.
- Compare data models using workplace access and maintenance needs.
- Save, reload, and verify CSV, JSON, and JSON Lines outputs.

## Scenario: Campus Operations Service Requests

A university operations office receives facilities, information technology, and public-safety requests. Employees need to answer different questions:

1. What is the current status of request `WO-1001`?
2. How many open requests belong to each agency?
3. Which building has the most urgent unresolved work?
4. Which employees are assigned to a request?
5. How did a request's status change over time?
6. What happens when an agency or building name changes?

These questions place different demands on the data. We will model the same domain in several ways and make the tradeoffs visible.

In [ ]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path
import json
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

OUTPUT_DIR = Path("week3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete. Output folder:", OUTPUT_DIR.resolve())

## Knowledge Point 1: A Dictionary Represents One Named Record

A Python dictionary stores key-value pairs. It is useful when fields have meaningful names and we need to work with one object at a time. Dictionary keys act like a small schema, but Python does not automatically guarantee that every record has the same keys or data types.

In [ ]:
work_order = {
    "request_id": "WO-1001",
    "created_at": "2026-09-20T08:30:00",
    "agency_id": "A01",
    "agency_name": "Facilities Operations",
    "location_id": "L01",
    "building": "Science Center",
    "category": "HVAC",
    "description": "Laboratory temperature above the approved range",
    "priority": "HIGH",
    "status": "OPEN",
    "estimated_cost": 850.00,
}

print("Request:", work_order["request_id"])
print("Current status:", work_order["status"])
print("Location:", work_order["building"])
print("Field names:", list(work_order.keys()))

### Safe Access, Updates, and Validation

Square brackets fail when a key is missing. The `.get()` method lets us provide a default. Updating a dictionary is easy, but an in-place update replaces the old value unless we separately preserve its history.

In [ ]:
print("Assigned employee:", work_order.get("assigned_to", "Not assigned"))

required_fields = {"request_id", "created_at", "agency_id", "location_id", "category", "status"}
missing_fields = required_fields - work_order.keys()
assert not missing_fields, f"Missing required fields: {missing_fields}"
assert work_order["status"] in {"OPEN", "IN_PROGRESS", "CLOSED"}
assert work_order["estimated_cost"] >= 0

work_order_updated = deepcopy(work_order)
work_order_updated["status"] = "IN_PROGRESS"
work_order_updated["assigned_to"] = "TECH-204"

print("Original status:", work_order["status"])
print("Updated status:", work_order_updated["status"])
print("Dictionary verification passed.")

## Knowledge Point 2: A List of Dictionaries Becomes a Table

A list preserves several records in order. A pandas DataFrame turns the records into rows and columns, making filtering, grouping, sorting, and aggregation convenient. This form works well for analysis when records share a consistent shape.

In [ ]:
work_orders = [
    {"request_id": "WO-1001", "created_at": "2026-09-20 08:30", "agency_id": "A01", "agency_name": "Facilities Operations", "location_id": "L01", "building": "Science Center", "category": "HVAC", "priority": "HIGH", "status": "OPEN", "estimated_cost": 850.00},
    {"request_id": "WO-1002", "created_at": "2026-09-20 09:10", "agency_id": "A02", "agency_name": "Information Technology", "location_id": "L02", "building": "Library", "category": "NETWORK", "priority": "HIGH", "status": "IN_PROGRESS", "estimated_cost": 300.00},
    {"request_id": "WO-1003", "created_at": "2026-09-20 10:45", "agency_id": "A01", "agency_name": "Facilities Operations", "location_id": "L03", "building": "West Hall", "category": "PLUMBING", "priority": "MEDIUM", "status": "CLOSED", "estimated_cost": 175.00},
    {"request_id": "WO-1004", "created_at": "2026-09-21 07:50", "agency_id": "A03", "agency_name": "Public Safety", "location_id": "L04", "building": "Parking Garage", "category": "LIGHTING", "priority": "HIGH", "status": "OPEN", "estimated_cost": 625.00},
    {"request_id": "WO-1005", "created_at": "2026-09-21 11:20", "agency_id": "A02", "agency_name": "Information Technology", "location_id": "L01", "building": "Science Center", "category": "CLASSROOM_TECH", "priority": "MEDIUM", "status": "OPEN", "estimated_cost": 450.00},
    {"request_id": "WO-1006", "created_at": "2026-09-22 08:05", "agency_id": "A01", "agency_name": "Facilities Operations", "location_id": "L02", "building": "Library", "category": "ELEVATOR", "priority": "CRITICAL", "status": "IN_PROGRESS", "estimated_cost": 2400.00},
    {"request_id": "WO-1007", "created_at": "2026-09-22 13:40", "agency_id": "A01", "agency_name": "Facilities Operations", "location_id": "L03", "building": "West Hall", "category": "CLEANING", "priority": "LOW", "status": "CLOSED", "estimated_cost": 90.00},
    {"request_id": "WO-1008", "created_at": "2026-09-23 15:15", "agency_id": "A03", "agency_name": "Public Safety", "location_id": "L04", "building": "Parking Garage", "category": "ACCESS_CONTROL", "priority": "MEDIUM", "status": "OPEN", "estimated_cost": 520.00},
]

orders_df = pd.DataFrame(work_orders)
orders_df["created_at"] = pd.to_datetime(orders_df["created_at"])

print("Shape:", orders_df.shape)
display(orders_df)

In [ ]:
open_orders = orders_df.loc[orders_df["status"] != "CLOSED"].copy()
open_by_agency = (
    open_orders.groupby(["agency_id", "agency_name"], as_index=False)
    .agg(open_requests=("request_id", "count"), estimated_open_cost=("estimated_cost", "sum"))
    .sort_values("open_requests", ascending=False)
)

display(open_by_agency)

critical_or_high = orders_df.loc[
    orders_df["priority"].isin(["CRITICAL", "HIGH"]) & (orders_df["status"] != "CLOSED"),
    ["request_id", "building", "category", "priority", "status"],
]
display(critical_or_high.sort_values(["priority", "request_id"]))

### The Flat-Table Duplication Problem

The table is convenient for analysis, but agency and building names repeat. If an agency is renamed, every matching row must be updated consistently. Repeated descriptive values also consume space and can develop spelling differences.

In [ ]:
duplication_summary = pd.DataFrame({
    "field": ["agency_name", "building"],
    "rows_stored": [orders_df["agency_name"].notna().sum(), orders_df["building"].notna().sum()],
    "unique_values": [orders_df["agency_name"].nunique(), orders_df["building"].nunique()],
})
duplication_summary["repeated_copies"] = duplication_summary["rows_stored"] - duplication_summary["unique_values"]
display(duplication_summary)

assert orders_df["request_id"].is_unique
assert orders_df["created_at"].notna().all()
assert orders_df["estimated_cost"].ge(0).all()
print("Flat-table verification passed.")

## Knowledge Point 3: Normalized Tables Separate Shared Entities

A relational design stores agencies and locations once, then refers to them by identifiers. This reduces inconsistent duplication and makes shared updates easier. The tradeoff is that a complete result often requires joins.

In [ ]:
agencies_df = orders_df[["agency_id", "agency_name"]].drop_duplicates().sort_values("agency_id").reset_index(drop=True)
locations_df = orders_df[["location_id", "building"]].drop_duplicates().sort_values("location_id").reset_index(drop=True)
requests_df = orders_df.drop(columns=["agency_name", "building"]).copy()

print("Agencies")
display(agencies_df)
print("Locations")
display(locations_df)
print("Requests with foreign-key-like identifiers")
display(requests_df.head())

### Join Tables to Reconstruct an Analytical View

The request table stores stable identifiers. Joins attach current descriptive values when a report needs them. We use `validate="many_to_one"` so pandas checks our relationship assumption.

In [ ]:
reconstructed_df = (
    requests_df
    .merge(agencies_df, on="agency_id", how="left", validate="many_to_one")
    .merge(locations_df, on="location_id", how="left", validate="many_to_one")
)

assert len(reconstructed_df) == len(requests_df)
assert reconstructed_df[["agency_name", "building"]].notna().all().all()
display(reconstructed_df[["request_id", "agency_name", "building", "category", "status"]])

### Many-to-Many Relationships Need a Junction Table

One work order may have several assigned employees, and one employee may work on several orders. Repeating employee columns such as `employee_1`, `employee_2`, and `employee_3` is brittle. A junction table represents the relationship directly.

In [ ]:
employees_df = pd.DataFrame([
    {"employee_id": "E101", "employee_role": "HVAC Technician"},
    {"employee_id": "E102", "employee_role": "Electrician"},
    {"employee_id": "E201", "employee_role": "Network Engineer"},
    {"employee_id": "E301", "employee_role": "Safety Officer"},
])

assignments_df = pd.DataFrame([
    {"request_id": "WO-1001", "employee_id": "E101"},
    {"request_id": "WO-1002", "employee_id": "E201"},
    {"request_id": "WO-1004", "employee_id": "E102"},
    {"request_id": "WO-1004", "employee_id": "E301"},
    {"request_id": "WO-1006", "employee_id": "E101"},
    {"request_id": "WO-1006", "employee_id": "E102"},
])

assignment_report = (
    assignments_df
    .merge(employees_df, on="employee_id", validate="many_to_one")
    .merge(requests_df[["request_id", "category", "priority"]], on="request_id", validate="many_to_one")
)
display(assignment_report.sort_values(["request_id", "employee_id"]))

## Knowledge Point 4: Nested JSON Keeps a Complete Object Together

JSON objects can nest dictionaries and lists. This gives strong locality when an application usually retrieves one complete request with its location, assignments, and history. The downside is duplication when shared entities appear inside many documents. Cross-document analysis may also require flattening or specialized queries.

In [ ]:
request_document = {
    "request_id": "WO-1004",
    "created_at": "2026-09-21T07:50:00",
    "category": "LIGHTING",
    "priority": "HIGH",
    "status": "OPEN",
    "agency": {
        "agency_id": "A03",
        "agency_name": "Public Safety",
    },
    "location": {
        "location_id": "L04",
        "building": "Parking Garage",
        "zone": "North Deck",
    },
    "assignments": [
        {"employee_id": "E102", "role": "Electrician"},
        {"employee_id": "E301", "role": "Safety Officer"},
    ],
    "status_history": [
        {"event_time": "2026-09-21T07:50:00", "status": "OPEN"},
        {"event_time": "2026-09-21T08:20:00", "status": "TRIAGED"},
    ],
}

print(json.dumps(request_document, indent=2))
print("Building:", request_document["location"]["building"])
print("Assigned employees:", [item["employee_id"] for item in request_document["assignments"]])

### Flatten Selected Nested Fields for Analysis

`pandas.json_normalize()` can flatten nested objects. Lists remain nested unless we explicitly expand them, which shows why analytical tables and application documents serve different needs.

In [ ]:
document_table = pd.json_normalize(request_document)
display(document_table.T.rename(columns={0: "value"}))

assignment_table = pd.json_normalize(
    request_document,
    record_path="assignments",
    meta=["request_id", ["location", "building"]],
)
display(assignment_table)

## Knowledge Point 5: Events Preserve What Changed

A current-state row tells us what is true now. An event tells us what happened at a particular time. Append-only events support auditability, debugging, replay, and incremental processing. Events should have stable identifiers so repeated delivery can be detected.

In [ ]:
events = [
    {"event_id": "EV-001", "request_id": "WO-1001", "event_time": "2026-09-20T08:30:00", "event_type": "REQUEST_CREATED", "value": "OPEN"},
    {"event_id": "EV-002", "request_id": "WO-1001", "event_time": "2026-09-20T09:00:00", "event_type": "ASSIGNED", "value": "E101"},
    {"event_id": "EV-003", "request_id": "WO-1001", "event_time": "2026-09-20T09:05:00", "event_type": "STATUS_CHANGED", "value": "IN_PROGRESS"},
    {"event_id": "EV-004", "request_id": "WO-1001", "event_time": "2026-09-20T14:35:00", "event_type": "NOTE_ADDED", "value": "Replacement sensor ordered"},
    {"event_id": "EV-005", "request_id": "WO-1001", "event_time": "2026-09-22T11:10:00", "event_type": "STATUS_CHANGED", "value": "CLOSED"},
    {"event_id": "EV-006", "request_id": "WO-1004", "event_time": "2026-09-21T07:50:00", "event_type": "REQUEST_CREATED", "value": "OPEN"},
    {"event_id": "EV-007", "request_id": "WO-1004", "event_time": "2026-09-21T08:20:00", "event_type": "STATUS_CHANGED", "value": "TRIAGED"},
]

events_df = pd.DataFrame(events)
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df.sort_values(["request_id", "event_time", "event_id"]).reset_index(drop=True)

assert events_df["event_id"].is_unique
display(events_df)

### Replay Events to Reconstruct Current State

The replay function processes events in order. It is intentionally small so that the relationship between event history and current state remains visible.

In [ ]:
def replay_request(request_id, event_table):
    request_events = (
        event_table.loc[event_table["request_id"] == request_id]
        .sort_values(["event_time", "event_id"])
    )
    state = {
        "request_id": request_id,
        "status": None,
        "assigned_employees": [],
        "notes": [],
        "last_event_time": None,
    }

    for event in request_events.to_dict("records"):
        if event["event_type"] in {"REQUEST_CREATED", "STATUS_CHANGED"}:
            state["status"] = event["value"]
        elif event["event_type"] == "ASSIGNED":
            if event["value"] not in state["assigned_employees"]:
                state["assigned_employees"].append(event["value"])
        elif event["event_type"] == "NOTE_ADDED":
            state["notes"].append(event["value"])
        state["last_event_time"] = event["event_time"]

    return state

state_1001 = replay_request("WO-1001", events_df)
state_1004 = replay_request("WO-1004", events_df)

print(state_1001)
print(state_1004)
assert state_1001["status"] == "CLOSED"
assert state_1001["assigned_employees"] == ["E101"]
assert state_1004["status"] == "TRIAGED"
print("Event replay verification passed.")

### Duplicate Delivery and Idempotence

Message and event systems may deliver a record more than once after a retry. A stable `event_id` lets a consumer remove exact duplicates before applying events.

In [ ]:
events_with_retry = pd.concat([events_df, events_df.iloc[[2]]], ignore_index=True)
print("Rows after simulated redelivery:", len(events_with_retry))
print("Duplicate event IDs:", events_with_retry["event_id"].duplicated().sum())

deduplicated_events = events_with_retry.drop_duplicates(subset="event_id", keep="first")
assert len(deduplicated_events) == len(events_df)
assert not deduplicated_events["event_id"].duplicated().any()
print("Deduplication verification passed.")

## Knowledge Point 6: Use SQL for Declarative Table Queries

A DataFrame is convenient inside a notebook. A relational database adds named tables, SQL queries, constraints, indexes, and transactions. Here we use an in-memory SQLite database to demonstrate how normalized tables can be queried without installing a server.

In [ ]:
with sqlite3.connect(":memory:") as connection:
    requests_df.to_sql("requests", connection, index=False, if_exists="replace")
    agencies_df.to_sql("agencies", connection, index=False, if_exists="replace")
    locations_df.to_sql("locations", connection, index=False, if_exists="replace")

    query = """
        SELECT
            a.agency_name,
            COUNT(*) AS open_requests,
            ROUND(SUM(r.estimated_cost), 2) AS estimated_open_cost
        FROM requests AS r
        JOIN agencies AS a ON r.agency_id = a.agency_id
        WHERE r.status <> 'CLOSED'
        GROUP BY a.agency_id, a.agency_name
        ORDER BY open_requests DESC, a.agency_name
    """
    sql_summary = pd.read_sql_query(query, connection)

display(sql_summary)
assert sql_summary["open_requests"].sum() == len(open_orders)

## Model Comparison

The best model follows the access pattern and maintenance requirements. A production system may legitimately use several forms: normalized operational tables as the source of record, JSON at an API boundary, events for history and integration, and a flat or star-shaped table for analytics.

In [ ]:
comparison = pd.DataFrame([
    {"structure": "Dictionary", "best_for": "One record in Python", "strength": "Named fields and simple access", "limitation": "No automatic schema or cross-record query"},
    {"structure": "Flat DataFrame", "best_for": "Analysis and export", "strength": "Filtering, grouping, and aggregation", "limitation": "Repeated descriptive data"},
    {"structure": "Normalized tables", "best_for": "Shared entities and transactions", "strength": "Consistency and relationship management", "limitation": "Joins are required"},
    {"structure": "Nested JSON", "best_for": "APIs and complete documents", "strength": "Related data has strong locality", "limitation": "Shared data may be duplicated"},
    {"structure": "Event log", "best_for": "History, integration, and replay", "strength": "Immutable audit trail", "limitation": "Current state must be derived"},
])
display(comparison)

### Decision Practice

Choose a structure for each requirement before displaying the suggested answer. More than one answer can be defensible if assumptions are stated.

In [ ]:
decisions = pd.DataFrame([
    {"requirement": "Send one complete request through an API", "suggested_structure": "Nested JSON", "reason": "The request and child objects travel together"},
    {"requirement": "Rename one agency consistently", "suggested_structure": "Normalized tables", "reason": "The shared agency name is stored once"},
    {"requirement": "Count requests by month and category", "suggested_structure": "Flat analytical table", "reason": "Grouping over consistent columns is efficient and clear"},
    {"requirement": "Explain every status change during an audit", "suggested_structure": "Event log", "reason": "The append-only history preserves changes"},
    {"requirement": "Validate one incoming record in a function", "suggested_structure": "Dictionary", "reason": "Named fields are easy to inspect and validate"},
])
display(decisions)

## Save, Reload, and Verify the Artifacts

A result is not reproducible merely because it appeared on screen. We save representative structures, reload them, and check that identifiers and counts survived the round trip. JSON Lines stores one JSON event per line and is useful for append-oriented records.

In [ ]:
flat_csv_path = OUTPUT_DIR / "work_orders_flat.csv"
document_json_path = OUTPUT_DIR / "request_document.json"
events_jsonl_path = OUTPUT_DIR / "request_events.jsonl"

orders_df.to_csv(flat_csv_path, index=False)
document_json_path.write_text(json.dumps(request_document, indent=2), encoding="utf-8")
events_df.to_json(events_jsonl_path, orient="records", lines=True, date_format="iso")

reloaded_flat = pd.read_csv(flat_csv_path, parse_dates=["created_at"])
reloaded_document = json.loads(document_json_path.read_text(encoding="utf-8"))
reloaded_events = pd.read_json(events_jsonl_path, orient="records", lines=True)

assert len(reloaded_flat) == len(orders_df)
assert set(reloaded_flat["request_id"]) == set(orders_df["request_id"])
assert reloaded_document["request_id"] == request_document["request_id"]
assert len(reloaded_events) == len(events_df)
assert reloaded_events["event_id"].is_unique

for path in [flat_csv_path, document_json_path, events_jsonl_path]:
    print(f"{path}: {path.stat().st_size:,} bytes")
print("Save/reload verification passed.")

## Verification Summary

The final cell gathers the major checks in one visible table. A workplace pipeline should make its assumptions and evidence similarly easy to review.

In [ ]:
verification = pd.DataFrame([
    {"check": "Unique request identifiers", "result": orders_df["request_id"].is_unique},
    {"check": "Valid request timestamps", "result": orders_df["created_at"].notna().all()},
    {"check": "Nonnegative estimated costs", "result": orders_df["estimated_cost"].ge(0).all()},
    {"check": "All agency joins matched", "result": reconstructed_df["agency_name"].notna().all()},
    {"check": "All location joins matched", "result": reconstructed_df["building"].notna().all()},
    {"check": "Unique event identifiers", "result": events_df["event_id"].is_unique},
    {"check": "CSV row count survived reload", "result": len(reloaded_flat) == len(orders_df)},
    {"check": "JSONL row count survived reload", "result": len(reloaded_events) == len(events_df)},
])
display(verification)
assert verification["result"].all()
print("All notebook verification checks passed.")

## Student Practice

Complete the following extension in new cells:

1. Add request `WO-1009` to the flat record list. Give it an existing agency and location.
2. Rebuild the DataFrame and verify that `request_id` remains unique.
3. Add one new employee and create two assignment relationships.
4. Create at least three events for `WO-1009`: creation, assignment, and status change.
5. Replay its events to reconstruct the current status.
6. Save and reload the revised event file.
7. Write 100–150 words recommending a primary structure for one workplace use case. State the access pattern, the selected structure, one benefit, one limitation, and how history will be preserved.

Do not use actual employee, customer, student, patient, or client information in the extension.

## AI-Assisted Learning Prompt

AI may help explain or review code only as allowed by the course policy. A useful review prompt is:

> I am using Google Colab and pandas. Review the following data-modeling code for a workplace service-request example. Identify duplicated data, missing validation, unsafe assumptions, and any relationship that should be represented with a separate table or event history. Do not rewrite the complete solution. Explain each recommendation so I can decide whether to apply it.

Students remain responsible for running the code, checking the result, documenting AI use, and explaining every submitted change.

## Key Takeaways

- Dictionaries are useful for named fields in one Python object.
- DataFrames make consistent records easy to inspect and analyze.
- Normalized tables reduce duplication and represent shared entities explicitly.
- Junction tables represent many-to-many relationships.
- Nested JSON provides locality for complete application objects and API messages.
- Event logs preserve change history and can reconstruct current state through replay.
- Stable identifiers support joins, deduplication, and idempotent processing.
- Real systems often use several representations because operational, integration, and analytical workloads differ.
- Saved outputs, reload tests, assertions, provenance, and privacy notes are part of the technical result—not optional documentation.